<a href="https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())
print("Unique content IDs:", df["content_id"].nunique())

print("\nDuplicate content IDs:")
print(df["content_id"].duplicated().sum())

print("\nDataset shape:")
print(df.shape)

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.
Rows: 30000
Columns: 44
Clients: 32
Unique content IDs: 30000

Duplicate content IDs:
0

Dataset shape:
(30000, 44)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
feature_cols = [
    "search_volume", "competition", "competition_level", "cpc",
    "content_type", "main_intent",
    "word_count", "char_count",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier"
]

label_cols = ["is_declining_label"]

context_cols = ["content_id", "client_id"]

excluded_cols = [
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Feature fields:", len(feature_cols))
print("Label fields:", label_cols)
print("Context fields:", context_cols)
print("Excluded fields:", excluded_cols)

all_classified = feature_cols + label_cols + context_cols + excluded_cols

print("\nDuplicate classifications:")
print(pd.Series(all_classified)[pd.Series(all_classified).duplicated()].tolist())

print("\nUnclassified columns:")
print([c for c in df.columns if c not in all_classified])

Feature fields: 31
Label fields: ['is_declining_label']
Context fields: ['content_id', 'client_id']
Excluded fields: ['trend_direction', 'trend_pct', 'provider_used', 'model_used', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Duplicate classifications:
[]

Unclassified columns:
['age_tier_order']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# 1. Grain check
grain_check = (
    df.groupby("content_id")
      .size()
      .reset_index(name="row_count")
)

print("Rows with duplicate content_id:")
print(grain_check[grain_check["row_count"] > 1].head())

# 2. Counts
print("\nTotal rows:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

# 3. Missing values
missing = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("missing_rate")
)

print("\nMissing-value rates:")
print(missing[missing["missing_rate"] > 0])

# 4. Missingness by content type
print("\nMissingness by content type:")
for col in [
    "search_volume",
    "competition",
    "cpc",
    "main_intent",
    "word_count",
    "char_count"
]:
    print(f"\n{col}")
    print(df.groupby("content_type")[col].apply(lambda x: x.isna().mean()))

# 5. Window-related checks
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("\nWindow column ranges:")
print(df[window_cols].agg(["min", "max"]))

# 6. Label distribution
print("\nLabel distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nDeclining rate:")
print((df["trend_direction"] == "down").mean())

Rows with duplicate content_id:
Empty DataFrame
Columns: [content_id, row_count]
Index: []

Total rows: 30000
Unique content items: 30000
Unique clients: 32

Missing-value rates:
                   missing_rate
provider_used          0.714600
word_count             0.256633
char_count             0.256633
word_count_tier        0.256633
char_count_tier        0.256633
model_used             0.191100
trend_pct              0.112933
competition_level      0.087000
search_volume          0.082267
cpc                    0.082267
competition            0.082267
main_intent            0.079133
scroll_rate            0.004167

Missingness by content type:

search_volume
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673
Name: search_volume, dtype: float64

competition
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673
Name: competition, dtype: float64

cpc
content_type
comparison arti

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Basic limits/checks

print("Clients:", df["client_id"].nunique())

print("\nRows per client:")
print(df.groupby("client_id").size().describe())

print("\nRows with missing keyword data:")
keyword_cols = ["search_volume", "competition", "competition_level", "cpc"]
print(df[keyword_cols].isna().all(axis=1).sum())

print("\nRows with missing word count:")
print(df["word_count"].isna().sum())

print("\nRows where avg_position == 0:")
print((df["avg_position"] == 0).sum())

print("\nRows with no previous 30-day impressions:")
print((df["impressions_prev_30d"] == 0).sum())

Clients: 32

Rows per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

Rows with missing keyword data:
2468

Rows with missing word count:
7699

Rows where avg_position == 0:
1205

Rows with no previous 30-day impressions:
3388


In [ ]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [ ]:
feature_cols.append("age_tier_order")

In [ ]:
from collections import Counter

all_classified = feature_cols + label_cols + context_cols + excluded_cols

duplicates = {
    col: count
    for col, count in Counter(all_classified).items()
    if count > 1
}

classified_not_in_data = [
    c for c in all_classified
    if c not in df.columns
]

unclassified = [
    c for c in df.columns
    if c not in set(all_classified)
]

print("Final column classification check")
print("=" * 40)

print("Features:", len(feature_cols))
print("Labels:", len(label_cols))
print("Context:", len(context_cols))
print("Excluded:", len(excluded_cols))

print("\nDuplicated classifications:")
print(duplicates)

print("\nClassified but not in dataset:")
print(classified_not_in_data)

print("\nUnclassified:")
print(unclassified)

print("\nTotal dataset columns:", len(df.columns))
print("Total classified unique columns:", len(set(all_classified)))

Final column classification check
Features: 32
Labels: 1
Context: 2
Excluded: 10

Duplicated classifications:
{}

Classified but not in dataset:
[]

Unclassified:
[]

Total dataset columns: 45
Total classified unique columns: 45


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.